# 03 — Camada Gold | CineData Analytics

**Objetivo:** modelagem dimensional (**Star Schema**) para o time de BI + respostas às perguntas de negócio.

```
                        dim_genres ── bridge_movie_genre ──┐
                        dim_people ── bridge_movie_person ──┤
                      dim_companies ─ bridge_movie_company ─┤
                                                           dim_movies ──── fact_movies_performance
                                                               └────────── dim_reviews (agregada por filme)
```

**Decisões de modelagem**
- **Chaves substitutas** (`sk_*`, `BIGINT`) geradas com `row_number()` sobre uma ordenação determinística da chave natural.
- **Grão da fato:** 1 linha por filme. As relações N:N (gênero, pessoa, produtora) ficam nas **tabelas-ponte**, nunca na fato — por isso nenhum join multiplica linhas da fato.
- A fato tem **1 linha por filme, para todos os filmes** da `dim_movies`; os filtros (ex.: só `Lançado`) são aplicados nas consultas analíticas. Para restringir a fato a filmes lançados, mude `APENAS_LANCADOS` para `True`.

## 1. Configuração

In [ ]:
from pyspark.sql import functions as F, Window, DataFrame

CATALOG = "workspace"
APENAS_LANCADOS = False

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

def sk(df: DataFrame, ordenar_por: list, nome_sk: str) -> DataFrame:
    """Surrogate key BIGINT determinística (1..N) a partir de uma ordenação estável da chave natural."""
    return df.withColumn(nome_sk, F.row_number().over(Window.orderBy(*ordenar_por)).cast("bigint"))

def salvar_gold(df: DataFrame, tabela: str) -> None:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    print(f"✔ {tabela}: {spark.table(tabela).count()} linhas")

## 2. Dimensões

In [ ]:
# dim_movies 
dim_movies = (
    sk(spark.table("silver.tb_info_filmes"), ["id_filme"], "sk_movie_id")
    .select(
        F.col("sk_movie_id"),
        F.col("id_filme").cast("string").alias("id_filme"),     # chave natural de origem
        "titulo",
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        "idioma_original",
        "status_filme",
        "sinopse",
    )
)
salvar_gold(dim_movies, "gold.dim_movies")

In [ ]:
# dim_genres (catálogo único de gêneros)
silver_generos = spark.table("silver.tb_generos")

dim_genres = (
    sk(silver_generos.select(F.col("genero").alias("nome_genero")).distinct(), ["nome_genero"], "sk_genre_id")
    .select("sk_genre_id", "nome_genero")
)
salvar_gold(dim_genres, "gold.dim_genres")

In [ ]:
# dim_people (Ator, Diretor, Roteirista)
# Mesma pessoa em papéis diferentes gera 1 linha por (nome, tipo): o papel faz parte do atributo da dimensão.
silver_pe = spark.table("silver.tb_pessoas_empresas")

dim_people = (
    sk(
        silver_pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
                 .select(F.col("nome").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
                 .distinct(),
        ["nome_pessoa", "tipo_pessoa"], "sk_person_id")
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)
salvar_gold(dim_people, "gold.dim_people")

# dim_companies (Produtoras)
dim_companies = (
    sk(
        silver_pe.filter(F.col("tipo_entidade") == "Produtora")
                 .select(F.col("nome").alias("nome_produtora")).distinct(),
        ["nome_produtora"], "sk_company_id")
    .select("sk_company_id", "nome_produtora")
)
salvar_gold(dim_companies, "gold.dim_companies")

In [ ]:
# dim_reviews (métrica resumida por filme)
reviews = spark.table("silver.tb_avaliacoes_usuarios")
mov_keys = spark.table("gold.dim_movies").select("sk_movie_id", "id_filme")

resumo = (
    reviews.join(mov_keys, "id_filme", "inner")
    .groupBy("sk_movie_id")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),   # AVG ignora notas NULL
    )
)
dim_reviews = (
    sk(resumo, ["sk_movie_id"], "sk_review_id")
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
salvar_gold(dim_reviews, "gold.dim_reviews")

## 3. Tabelas-ponte (bridge tables)

In [ ]:
dm = spark.table("gold.dim_movies").select("sk_movie_id", "id_filme")

# movie <-> genre
bridge_movie_genre = (
    spark.table("silver.tb_generos").join(dm, "id_filme")
    .join(spark.table("gold.dim_genres"), F.col("genero") == F.col("nome_genero"))
    .select("sk_movie_id", "sk_genre_id").distinct()
)
salvar_gold(bridge_movie_genre, "gold.bridge_movie_genre")

# movie <-> person
bridge_movie_person = (
    spark.table("silver.tb_pessoas_empresas").join(dm, "id_filme")
    .join(spark.table("gold.dim_people"),
          (F.col("nome") == F.col("nome_pessoa")) & (F.col("tipo_entidade") == F.col("tipo_pessoa")))
    .select("sk_movie_id", "sk_person_id").distinct()
)
salvar_gold(bridge_movie_person, "gold.bridge_movie_person")

# movie <-> company
bridge_movie_company = (
    spark.table("silver.tb_pessoas_empresas").filter(F.col("tipo_entidade") == "Produtora").join(dm, "id_filme")
    .join(spark.table("gold.dim_companies"), F.col("nome") == F.col("nome_produtora"))
    .select("sk_movie_id", "sk_company_id").distinct()
)
salvar_gold(bridge_movie_company, "gold.bridge_movie_company")

## 4. Tabela Fato — `gold.fact_movies_performance`

Grão: **1 filme**. As tabelas Silver de origem já são únicas por `id_filme`, então os `LEFT JOIN`s não multiplicam linhas (isto é verificado na seção de validação).

In [ ]:
dim_mov = spark.table("gold.dim_movies")
base = dim_mov.filter(F.col("status_filme") == "Lançado") if APENAS_LANCADOS else dim_mov

fin = spark.table("silver.tb_financeiro_filmes")
met = spark.table("silver.tb_metricas_engajamento")

dec = lambda c: F.col(c).cast("decimal(18,2)")

fact = (
    base.select("sk_movie_id", "id_filme")
    .join(fin, "id_filme", "left")
    .join(met, "id_filme", "left")
    .select(
        "sk_movie_id",
        dec("orcamento_usd").alias("orcamento_usd"),
        dec("receita_usd").alias("receita_usd"),
        dec("lucro_usd").alias("lucro_usd"),
        dec("orcamento_brl").alias("orcamento_brl"),
        dec("receita_brl").alias("receita_brl"),
        dec("lucro_brl").alias("lucro_brl"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb"),
    )
)
salvar_gold(fact, "gold.fact_movies_performance")

## 5. Validação do modelo (PKs únicas e integridade referencial)

In [ ]:
def pk_unica(tabela, pk):
    df = spark.table(tabela); t, d = df.count(), df.select(pk).distinct().count()
    print(f"{'✔' if t == d else '✘'} PK única   {tabela:<34} {pk}")
    assert t == d, f"PK duplicada em {tabela}"

def fk_valida(tabela, fk, ref, pk):
    # Colunas renomeadas antes do join: evita referência ambígua entre DataFrames (exigência do Spark Connect)
    filha = spark.table(tabela).select(F.col(fk).alias("_fk"))
    pai   = spark.table(ref).select(F.col(pk).alias("_pk"))
    orfas = filha.join(pai, F.col("_fk") == F.col("_pk"), "left_anti").count()
    print(f"{'✔' if orfas == 0 else '✘'} FK válida  {tabela}.{fk} -> {ref}.{pk}  (órfãs: {orfas})")
    assert orfas == 0, f"FK órfã em {tabela}.{fk}"

for t, pk in [("gold.dim_movies", "sk_movie_id"), ("gold.dim_genres", "sk_genre_id"), ("gold.dim_people", "sk_person_id"),
              ("gold.dim_companies", "sk_company_id"), ("gold.dim_reviews", "sk_review_id"),
              ("gold.fact_movies_performance", "sk_movie_id")]:
    pk_unica(t, pk)

fk_valida("gold.fact_movies_performance", "sk_movie_id", "gold.dim_movies", "sk_movie_id")
fk_valida("gold.dim_reviews", "sk_movie_id", "gold.dim_movies", "sk_movie_id")
fk_valida("gold.bridge_movie_genre",   "sk_movie_id",   "gold.dim_movies",    "sk_movie_id")
fk_valida("gold.bridge_movie_genre",   "sk_genre_id",   "gold.dim_genres",    "sk_genre_id")
fk_valida("gold.bridge_movie_person",  "sk_movie_id",   "gold.dim_movies",    "sk_movie_id")
fk_valida("gold.bridge_movie_person",  "sk_person_id",  "gold.dim_people",    "sk_person_id")
fk_valida("gold.bridge_movie_company", "sk_movie_id",   "gold.dim_movies",    "sk_movie_id")
fk_valida("gold.bridge_movie_company", "sk_company_id", "gold.dim_companies", "sk_company_id")

## 6. Desafio de Analytics

Para as perguntas 5 e 6 a **data de referência** é a maior data de lançamento da base, **ignorando datas futuras e filmes não lançados**. Os recortes "últimos 2 / 5 anos" são `[data_ref − N anos, data_ref]`.

In [ ]:
# Data de referência (maior lançamento real, sem futuros nem não-lançados)
data_ref = spark.sql("""
    SELECT MAX(data_lancamento) AS data_ref
    FROM gold.dim_movies
    WHERE status_filme = 'Lançado' AND data_lancamento <= CURRENT_DATE()
""").first()["data_ref"]
print("Data de referência:", data_ref)

In [ ]:
# 1. Receita total (R$) de todos os filmes da base
display(spark.sql("""
    SELECT ROUND(SUM(receita_brl), 2) AS receita_total_brl
    FROM gold.fact_movies_performance
"""))

In [ ]:
# 2. Top 5 filmes por popularidade
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

In [ ]:
# 3. Quantidade de filmes por gênero (maior -> menor)
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
"""))

In [ ]:
# 4. Top 10 filmes por receita, com RANK()
display(spark.sql("""
    SELECT posicao_ranking, titulo, receita_usd, receita_brl
    FROM (
        SELECT RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking,
               m.titulo, f.receita_usd, f.receita_brl
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    ORDER BY posicao_ranking, titulo
    LIMIT 10
"""))

In [ ]:
# 5. Ator com mais participações em filmes lançados nos últimos 2 anos (empates aparecem juntos)
display(spark.sql(f"""
    SELECT ator, qtd_filmes FROM (
        SELECT p.nome_pessoa AS ator,
               COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes,
               RANK() OVER (ORDER BY COUNT(DISTINCT m.sk_movie_id) DESC) AS rk
        FROM gold.bridge_movie_person b
        JOIN gold.dim_people p ON p.sk_person_id = b.sk_person_id
        JOIN gold.dim_movies m ON m.sk_movie_id  = b.sk_movie_id
        WHERE p.tipo_pessoa = 'Ator'
          AND m.status_filme = 'Lançado'
          AND m.data_lancamento BETWEEN ADD_MONTHS(DATE'{data_ref}', -24) AND DATE'{data_ref}'
        GROUP BY p.nome_pessoa
    )
    WHERE rk = 1
    ORDER BY ator
"""))

In [ ]:
# 6. Produtora com maior lucro nos últimos 5 anos (lucro de cada filme é atribuído a cada produtora participante)
display(spark.sql(f"""
    SELECT produtora, lucro_total_usd, lucro_total_brl FROM (
        SELECT c.nome_produtora AS produtora,
               SUM(f.lucro_usd) AS lucro_total_usd,
               SUM(f.lucro_brl) AS lucro_total_brl,
               RANK() OVER (ORDER BY SUM(f.lucro_brl) DESC) AS rk
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m            ON m.sk_movie_id = f.sk_movie_id
        JOIN gold.bridge_movie_company bc ON bc.sk_movie_id = f.sk_movie_id
        JOIN gold.dim_companies c         ON c.sk_company_id = bc.sk_company_id
        WHERE m.data_lancamento BETWEEN ADD_MONTHS(DATE'{data_ref}', -60) AND DATE'{data_ref}'
          AND f.lucro_brl IS NOT NULL
        GROUP BY c.nome_produtora
    )
    WHERE rk = 1
"""))